# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Here we print record sets and for each, list its fields with their corresponding `@id`s as defined by the Croissant schema.

In [ ]:
# List all available record sets with their `@id`s and contained field IDs
for rs in dataset.metadata.record_sets:
    print(f"Record set: {rs.name} [@id={rs.id}]")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} [@id={field.id}] (column @id: {field.column.id if getattr(field, 'column', None) else 'none'})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview. Here, we demonstrate loading all available record sets into pandas DataFrames. You may select the relevant record set(s) for further processing.

In [ ]:
# Extract data from each record set
record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"Loaded {len(df)} records from record set @id={record_set}")
    print(f"Schema for @id={record_set}: {df.columns.tolist()}")
    print()

# Choose one record set for further analysis (the main data table)
selected_record_set = None
if record_sets:
    selected_record_set = record_sets[0]  # first as default
    print(f"Selected record set: {selected_record_set}")
    print(dataframes[selected_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

This section includes operations like removing outliers, transforming data distributions, and grouping data by key attributes to prepare for further analysis. All field names must use their exact `@id` as shown previously.

In [ ]:
# For illustration, we will process the first record set (edit as needed)
if selected_record_set is not None:
    df = dataframes[selected_record_set].copy()
    print(f"Columns in dataframe: {df.columns.tolist()}")

    # Attempt to pick a numeric field (float or int) by heuristic (edit as needed for your case)
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        # Fallback: try to coerce columns with likely numeric names
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Selected numeric field for analysis: {numeric_field}")
        threshold = df[numeric_field].mean()  # You can set your threshold

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (total {len(filtered_df)} records):")
        print(filtered_df.head())

        # Normalize
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, normalized_col]].head())

        # Attempt to find a suitable group field, e.g. 'sample' or similar
        group_field = None
        for candidate in df.columns:
            if candidate != numeric_field and df[candidate].dtype == object and df[candidate].nunique() < len(df) / 2:
                group_field = candidate
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No obvious categorical field to group by was found.")
    else:
        print("No numeric field found to perform EDA.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we plot a histogram and scatterplot if at least two numeric columns are available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set is not None and numeric_candidates:
    df = dataframes[selected_record_set].copy()
    # Plot histogram for the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If at least 2 numeric fields, show scatterplot
    if len(numeric_candidates) >= 2:
        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=df[numeric_candidates[0]], y=df[numeric_candidates[1]])
        plt.xlabel(numeric_candidates[0])
        plt.ylabel(numeric_candidates[1])
        plt.title(f'{numeric_candidates[0]} vs {numeric_candidates[1]}')
        plt.show()
else:
    print("Not enough numeric data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the Croissant-annotated dataset and displayed its structure using `mlcroissant`.
- We identified available record sets and fields (referenced by their `@id`s for clarity).
- We loaded the primary data into DataFrames, filtered and normalized key numeric columns, and performed basic grouping and aggregation.
- Visualized distributions and relationships for quantitative inspection.

Further steps could include more advanced statistical testing, model fitting, or deep dives into domain-specific questions relevant to mass spectrometry data.